# Sunside — Visual Factory (Colab)

Фотореалізм · NSFW без цензури · персонажі · кастомний розмір під лендінги

### Запуск
1. **Runtime → GPU** → **Restart session**
2. **Run all** → чекай **`App started successful`**
3. Відкрий **`https://….gradio.live`**

### Simple flow
1. (опційно) **Character** → Aria / Mila / Vera  
2. Завантаж **Face ref** (еталон анфас) + галочка **Face lock**  
3. **Scenario** або один **Sunside** style  
4. **Size** → промпт сцени → **Generate**

### Face lock
Працює **після** генерації (Inswapper): спочатку кадр, потім підміна обличчя з UI-аплоаду.  
Старий FaceSwap у Image Prompt вимкнено (краш на Colab).

### Вже всередині
- Модель: **CyberRealistic XL** (+ emotional / face / body LoRA)
- Лише Sunside-стилі (+ Semi Realistic)
- Black Out NSFW вимкнено

⚠️ Код **-9** → Restart session → Run all  
⚠️ **Лише 1** Sunside-стиль за раз.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
import zipfile

os.chdir("/content")
os.makedirs("/content", exist_ok=True)

REPO = "/content/Fooocus"
REPO_URL = "https://github.com/sunsideaspect/foocus_sunside.git"
ZIP_URL = "https://github.com/sunsideaspect/foocus_sunside/archive/refs/heads/main.zip"

print("=" * 60)
print("КРОК 1/3 — Код foocus_sunside")
print("=" * 60)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pygit2==1.15.1"], check=True)


def remove_repo(path: str) -> None:
    if os.path.exists(path):
        subprocess.run(["rm", "-rf", path], check=False)


def repo_ok(path: str) -> bool:
    return os.path.isfile(os.path.join(path, "launch.py"))


def clone_with_git() -> bool:
    os.chdir("/content")
    remove_repo(REPO)
    print("→ git clone...")
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO],
        capture_output=True,
        text=True,
        cwd="/content",
    )
    if r.returncode != 0:
        print("  git stderr:", (r.stderr or "").strip())
        return False
    return repo_ok(REPO)


def clone_with_zip() -> bool:
    os.chdir("/content")
    remove_repo(REPO)
    zip_path = "/content/foocus_sunside_main.zip"
    print("→ ZIP fallback...")
    r = subprocess.run(
        ["wget", "--progress=dot:giga", "-O", zip_path, ZIP_URL],
        cwd="/content",
    )
    if r.returncode != 0:
        return False
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content")
    extracted = "/content/foocus_sunside-main"
    if not os.path.isdir(extracted):
        return False
    shutil.move(extracted, REPO)
    return repo_ok(REPO)


ok = False
for attempt in range(1, 4):
    if clone_with_git():
        ok = True
        print(f"✅ Код завантажено (git, спроба {attempt})")
        break
    print(f"  Повтор git через 8 с ({attempt}/3)...")
    time.sleep(8)

if not ok:
    ok = clone_with_zip()
    if ok:
        print("✅ Код завантажено (ZIP)")

if not ok:
    raise RuntimeError("Не вдалось завантажити. Runtime → Restart session → Run all")

os.chdir(REPO)
os.environ["TORCH_COMMAND"] = (
    "pip install torch torchvision torchaudio "
    "--index-url https://download.pytorch.org/whl/cu124"
)

print("\n" + "=" * 60)
print("Перевірка GPU")
print("=" * 60)
gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_check.returncode != 0 or "GPU" not in (gpu_check.stdout or ""):
    raise RuntimeError("❌ GPU не знайдено. Runtime → GPU → Restart session → Run all")
print("✅", (gpu_check.stdout or "").strip().split("\n")[0])
print("→ Далі: КРОК 2 (модель) + КРОК 3 (launch)")

In [ ]:
# @title ⚙️ Preset (CyberRealistic only)
SELECTED_PRESET = "realistic_cyberrealistic_xl"
model_choice = "CyberRealistic XL"
print(f"✅ Product preset: {SELECTED_PRESET}")
print("💡 Character → Scenario/Style → Size → Prompt → Generate")
print("💡 Інші моделі прибрані з product UI")

In [ ]:
import os
import re
import stat
import subprocess
import sys

from IPython.display import HTML, display

if "SELECTED_PRESET" not in globals():
    SELECTED_PRESET = "realistic_cyberrealistic_xl"
    model_choice = "CyberRealistic (default — спочатку КРОК 2)"

os.chdir("/content")
os.makedirs("/content", exist_ok=True)

REPO = "/content/Fooocus"
if not os.path.isfile(os.path.join(REPO, "launch.py")):
    raise RuntimeError("Спочатку запусти КРОК 1 (клон репозиторію).")
os.chdir(REPO)

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["LAUNCH_LIVE_OUTPUT"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Hugging Face: на Colab urllib часто 403 на великих LFS — hub ок; Xet інколи висить
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
        print("✅ HF_TOKEN з Colab Secrets підключено")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    print("💡 urllib 403 на великих моделях — нормально. Чекай рядок [download] huggingface_hub…")
    print("💡 Якщо hub теж падає з 403 — додай Secret HF_TOKEN (read) на huggingface.co/settings/tokens")


def gpu_total_vram_mb() -> int:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=True,
    )
    return int((r.stdout or "").strip().split("\n")[0])


vram_mb = gpu_total_vram_mb()
vram_flag = "--always-high-vram" if vram_mb >= 22000 else "--always-normal-vram"

print("=" * 60)
print("КРОК 3/3 — Запуск Sunside Product")
print("=" * 60)
print(f"Preset: {SELECTED_PRESET}")
print(f"Model:  {model_choice}")
print(f"VRAM:   {vram_mb} MB → {vram_flag}")
if vram_mb < 16000:
    print("⚠️ T4: normal-vram. Якщо -9 → Restart session → Run all.")
print("\nЧекай: torch → моделі → Running on public URL → App started successful\n")


def _bar_html(pct: int, detail: str) -> str:
    pct = max(0, min(100, pct))
    safe = detail.replace("&", "&amp;").replace("<", "&lt;")
    return (
        f'<div style="margin:6px 0 10px">'
        f'<div style="background:#1e293b;border-radius:8px;height:26px;overflow:hidden;border:1px solid #334155">'
        f'<div style="width:{pct}%;height:100%;background:linear-gradient(90deg,#16a34a,#4ade80);'
        f'transition:width .4s"></div></div>'
        f'<div style="font:12px/1.4 monospace;color:#94a3b8;margin-top:4px">{pct}% — {safe}</div></div>'
    )


def _emit_progress(line: str, bar_holder: dict) -> bool:
    m = re.search(r"(\d+)%", line)
    if not m or not ("|" in line or "G/" in line or "M/" in line or "kB" in line):
        return False
    pct = int(m.group(1))
    html = _bar_html(pct, line.strip()[:140])
    if bar_holder.get("disp") is None:
        bar_holder["disp"] = display(HTML(html), display_id=True)
    else:
        bar_holder["disp"].update(HTML(html))
    return True


def run_live(argv, cwd=REPO, label=""):
    if label:
        print(label, flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["LAUNCH_LIVE_OUTPUT"] = "1"
    p = subprocess.Popen(
        argv,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    assert p.stdout is not None
    bar_holder: dict = {}
    partial = b""
    last_lines: list = []
    while True:
        chunk = p.stdout.read(1)
        if not chunk:
            if p.poll() is not None:
                break
            continue
        if chunk in (b"\r", b"\n"):
            line = partial.decode("utf-8", errors="ignore").strip()
            partial = b""
            if not line:
                continue
            last_lines.append(line)
            if len(last_lines) > 40:
                last_lines.pop(0)
            if not _emit_progress(line, bar_holder):
                print(line, flush=True)
        else:
            partial += chunk
    tail = partial.decode("utf-8", errors="ignore").strip()
    if tail:
        last_lines.append(tail)
        if not _emit_progress(tail, bar_holder):
            print(tail, flush=True)
    rc = p.wait()
    if rc != 0:
        tip = "\n--- останні рядки логу ---\n" + "\n".join(last_lines[-25:])
        if rc in (-9, 137):
            raise RuntimeError(
                "OOM (код -9). Runtime → Restart session → Run all. "
                "На T4: Cyber / Juggernaut / RealVis." + tip
            )
        raise RuntimeError(
            f"Команда завершилась з кодом {rc}: {' '.join(argv)}\n"
            "↑ див. Traceback вище / блок нижче. "
            "Часто: бите завантаження моделі → Restart session → Run all."
            + tip
        )


print("→ A: numpy...")
run_live(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "numpy==1.26.4"],
    cwd="/content",
)
print("→ B: gradio...")
run_live(
    [sys.executable, "-m", "pip", "install",
     "starlette>=0.27.0,<1.0.0", "fastapi>=0.100.0,<0.115.0", "gradio==3.41.2"],
    cwd=REPO,
)
print("→ B1: Face lock deps (insightface)...")
run_live(
    [sys.executable, "-m", "pip", "install", "-q",
     "insightface==0.7.3", "onnxruntime-gpu==1.17.1", "opencv-python-headless"],
    cwd="/content",
)
# Xet CDN на Colab дає 403 SignatureError — прибираємо hf_xet
print("→ B2: disable Hugging Face Xet (Colab 403 fix)...")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "hf_xet", "hf-xet"],
    check=False,
)


def gradio_pkg_dir() -> str:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", "gradio"],
        capture_output=True,
        text=True,
        check=True,
    )
    for line in r.stdout.splitlines():
        if line.startswith("Location:"):
            return os.path.join(line.split(":", 1)[1].strip(), "gradio")
    raise RuntimeError("gradio not found after pip install")


print("→ C: frpc Share...")
frpc = os.path.join(gradio_pkg_dir(), "frpc_linux_amd64_v0.2")
if not (os.path.isfile(frpc) and os.path.getsize(frpc) > 1_000_000):
    subprocess.run(
        ["wget", "--progress=dot:giga", "-O", frpc,
         "https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64"],
        check=True,
    )
    os.chmod(frpc, stat.S_IRWXU)

# Прибрати недокачані файли (частий код 1 після обриву download)
MIN_OK = {
    "CyberRealisticXLPlay_V10.0_FP16.safetensors": 6_000_000_000,
    "juggernautXL_ragnarokBy.safetensors": 6_000_000_000,
    "epicrealismXL_pureFix.safetensors": 6_000_000_000,
    "RealVisXL_V5.0_fp16.safetensors": 6_000_000_000,
    "emotional.safetensors": 1_000_000,
    "face_xl_v0_1.safetensors": 50_000_000,
    "10mb_bodyproportion.safetensors": 1_000_000,
}
preset_path = os.path.join(REPO, "presets", f"{SELECTED_PRESET}.json")
if not os.path.isfile(preset_path):
    raise RuntimeError(
        f"Немає preset-файлу: {preset_path}. "
        "Restart session → Run all (онови код)."
    )
print(f"✅ Preset file: {preset_path}")
for root_name in ("checkpoints", "loras"):
    root = os.path.join(REPO, "models", root_name)
    if not os.path.isdir(root):
        continue
    for dirpath, _, files in os.walk(root):
        for fn in files:
            need = MIN_OK.get(fn)
            if need is None:
                continue
            fp = os.path.join(dirpath, fn)
            sz = os.path.getsize(fp)
            if sz < need:
                print(f"⚠️ Битий/неповний файл ({sz} < {need}), видаляю: {fp}")
                os.remove(fp)

launch_args = [
    sys.executable, "-u", "launch.py",
    "--preset", SELECTED_PRESET,
    "--disable-censor",
    "--disable-pro-mode",
    "--disable-preset-selection",
    "--share",
    vram_flag,
    "--disable-in-browser",
]
# Product mode is ON by default; pass env for clarity
os.environ["SUNSIDE_PRODUCT"] = "1"

print("\n" + "=" * 60)
print("ЗАПУСК (лог у реальному часі)")
print("=" * 60 + "\n")

os.chdir(REPO)
run_live(launch_args, cwd=REPO)
print("\n✅ Сесію зупинено (нормально, якщо ти сам вимкнув).")